# 🧭 مركز تحكم شام — يتعرّف تلقائياً على كل الدفاتر والنتائج

**ماذا يفعل؟** لا تحتاجين تذكّر أي شيء. هذا الدفتر:
1. يجد **كل** دفاترك على Kaggle (حتى التي أسماؤها أرقام مثل `notebook24ffaf0b22`) ويقرأ محتواها ليعرف **ما هو كل دفتر** (نص / صورة / صوت / فيديو / المرحلة الثانية…).
2. يقرأ حالة آخر تشغيل لكل دفتر: ✅ اكتمل / ❌ فشل (مع سبب الفشل) / ⏳ يعمل.
3. يكتشف النسخ **المكررة** من نفس المسار ويحدد الأساسي منها.
4. يقرأ مجموعات البيانات (النتائج المحفوظة) ومقاييس التقدم.
5. يحسب **الخطوة الحالية** بوضوح ويقول لك ماذا تفعلين الآن بالضبط.
6. **التقرير الكامل** (أسماء الدفاتر الحقيقية + روابطها) يصلك **أنتِ وحدك** على تيليجرام.
7. **نسخة مختصرة بلا أسماء** (مثل «مسار ترميز الصورة 1»: الحالة، التقدم، الخطأ فقط) تُحفظ ليقرأها Claude — لا يطّلع على أسماء الدفاتر ولا محتواها ولا روابطها.

**الإعداد لمرة واحدة:**
- Accelerator: **None** (لا يحتاج GPU)، Internet: **On**.
- Add-ons → Secrets: نفس الأسرار الموجودة أصلاً: `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`, `TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID`.
- ثم **Save & Run All**. يمكن جدولته (Schedule) مرة يومياً ليصلك التقرير كل يوم.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    # جلسة مفتوحة أُعيد تشغيلها: نجلب آخر نسخة من الكود بالتوكن (الرابط المحفوظ بلا توكن عمداً)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "--depth", "1", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
# أمان: إزالة التوكن من رابط git المحفوظ داخل نتاج الجلسة
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)
subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)

os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
print("جاهز.")

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, "/kaggle/working/Ttbik/ai-system/scripts")
# إعادة التشغيل في نفس الجلسة: نُسقط النسخ القديمة المحمّلة في الذاكرة ليُستخدم أحدث كود
for _m in ("sham_registry", "kaggle_auto_resume"):
    sys.modules.pop(_m, None)
from kaggle_auto_resume import _load_kaggle_api
from sham_registry import discover, build_registry, render_report, assign_aliases, public_registry, render_public_report

api = _load_kaggle_api()
kernels, datasets = discover(api, Path("/kaggle/working/_registry_tmp"))
registry = build_registry(kernels, datasets)
assign_aliases(registry)
Path("/kaggle/working/sham_registry_FULL_private.json").write_text(json.dumps(registry, ensure_ascii=False, indent=2), encoding="utf-8")
report = render_report(registry)
print(report)

# النسخة العامة (بلا أسماء ولا روابط) — الوحيدة التي تغادر حسابك
public = public_registry(registry)
public_report = render_public_report(public)

In [ ]:
# إرسال التقرير إلى تيليجرام (يُقسَّم تلقائياً إن كان طويلاً)
sys.path.insert(0, "/kaggle/working/Ttbik/ai-system/colab/sham_small")
from telegram_report import send_telegram_message
try:
    tg_token = secrets.get_secret("TELEGRAM_BOT_TOKEN")
    tg_chat = secrets.get_secret("TELEGRAM_CHAT_ID")
    chunk, parts = "", []
    for line in report.splitlines():
        if len(chunk) + len(line) > 3500:
            parts.append(chunk); chunk = ""
        chunk += line + "\n"
    parts.append(chunk)
    for p in parts:
        send_telegram_message(tg_token, tg_chat, p)
    print("أُرسل التقرير إلى تيليجرام.")
except Exception as exc:
    print("لم يُرسل إلى تيليجرام:", exc)
    if "No user secrets exist" in str(exc):
        print("👉 الحل: من Add-ons ← Secrets فعّلي (ضعي علامة ✓ بجانب) TELEGRAM_BOT_TOKEN و TELEGRAM_CHAT_ID لهذا الدفتر، ثم أعيدي التشغيل.")

In [ ]:
# حفظ النسخة العامة فقط في GitHub (فرع sham-status) — بلا أسماء دفاتر ولا روابط ولا محتوى.
# الأسماء الكاملة تصلك أنتِ وحدك عبر تيليجرام.
import shutil, tempfile
work = Path(tempfile.mkdtemp())
url = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", "sham-status", url, str(work / "s")], capture_output=True)
repo = work / "s"
if r.returncode != 0:  # أول مرة: فرع جديد مستقل بلا أي كود
    repo.mkdir(parents=True)
    subprocess.run(["git", "init", "-q", "-b", "sham-status"], cwd=repo, check=True)
    subprocess.run(["git", "remote", "add", "origin", url], cwd=repo, check=True)
# فرع التقارير لا يحوي كود الموقع — نمنع فيرسيل من محاولة بنائه (كان يرسل رسائل "Deployment failed")
(repo / "vercel.json").write_text(json.dumps({"git": {"deploymentEnabled": False}}), encoding="utf-8")
(repo / "sham-registry.json").write_text(json.dumps(public, ensure_ascii=False, indent=2), encoding="utf-8")
(repo / "sham-report.txt").write_text(public_report, encoding="utf-8")
subprocess.run(["git", "-c", "user.name=sham-control-center", "-c", "user.email=sham@ttbik.local", "add", "-A"], cwd=repo, check=True)
c = subprocess.run(["git", "-c", "user.name=sham-control-center", "-c", "user.email=sham@ttbik.local", "commit", "-q", "-m", "sham registry update"], cwd=repo)
if c.returncode == 0:
    subprocess.run(["git", "push", "-q", "origin", "sham-status"], cwd=repo, check=True)
    print("حُفظ السجل في GitHub (فرع sham-status).")
else:
    print("لا تغيير منذ آخر تقرير.")
shutil.rmtree(work, ignore_errors=True)